<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/enzo%2Fp10-9-product-consolidation-ai/P10_9_REAL_CHECKPOINT_SMOKE_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P10.9 — Smoke real de checkpoints P10.6 + P10.7

Este notebook **no entrena** y no accede a datasets ni tests sellados.
Solo verifica los dos `.pt` congelados y ejecuta un forward sintético por modelo.

AI Module fijado al commit:

`a967f94b48b1f3525dccc1faf4431221b2b1522a`

Checkpoints esperados:

- P10.6 subarticular: `d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a`
- P10.7 disc multitask: `16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293`

Ejecutar con **CPU**. No hace falta GPU.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib
import json

P10_6 = Path(
    "/content/drive/MyDrive/PFI_MVP/models/"
    "P10_6_rsna_findings/subarticular_axial_t2_2p5d/"
    "final_internal_test_evaluation/frozen_subarticular_checkpoint.pt"
)
P10_7 = Path(
    "/content/drive/MyDrive/PFI_MVP/models/"
    "P10_7_spider_degenerative/final_internal_test_evaluation/"
    "frozen_p10_7_spider_degenerative_multitask.pt"
)

EXPECTED = {
    str(P10_6): "d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a",
    str(P10_7): "16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293",
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

audit = []
for raw, expected in EXPECTED.items():
    path = Path(raw)
    if not path.is_file():
        raise FileNotFoundError(path)
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(
            f"ABORT: hash inesperado para {path.name}: {actual}"
        )
    audit.append({
        "fileName": path.name,
        "sizeBytes": path.stat().st_size,
        "sha256": actual,
        "hashMatch": True,
        "trainingExecuted": False,
    })

print(json.dumps(audit, indent=2, ensure_ascii=False))
print("CHECKPOINT_PREFLIGHT_OK")


[
  {
    "fileName": "frozen_subarticular_checkpoint.pt",
    "sizeBytes": 48655231,
    "sha256": "d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a",
    "hashMatch": true,
    "trainingExecuted": false
  },
  {
    "fileName": "frozen_p10_7_spider_degenerative_multitask.pt",
    "sizeBytes": 22182638,
    "sha256": "16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293",
    "hashMatch": true,
    "trainingExecuted": false
  }
]
CHECKPOINT_PREFLIGHT_OK


In [3]:
import os, shutil, subprocess
from pathlib import Path

REPO = Path("/content/pfi_ai_p10_9")
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    [
        "git", "clone", "--quiet",
        "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git",
        str(REPO),
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--quiet", "a967f94b48b1f3525dccc1faf4431221b2b1522a"],
    check=True,
)

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()

if head != "a967f94b48b1f3525dccc1faf4431221b2b1522a":
    raise RuntimeError(f"Commit inesperado: {head}")

print("AI_MODULE_SHA_OK:", head)


AI_MODULE_SHA_OK: a967f94b48b1f3525dccc1faf4431221b2b1522a


In [4]:
# Solo dependencias de runtime necesarias para cargar EfficientNet.
# No se instala ni ejecuta ningún código de entrenamiento.
!pip -q install "timm==1.0.28" "pydantic>=2.7,<3"


In [5]:
import os
import subprocess
from pathlib import Path

OUTPUT = Path(
    "/content/drive/MyDrive/PFI_MVP/results/"
    "P10_9_product_checkpoint/REAL_CHECKPOINT_SMOKE.json"
)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "ai_service")
env["PFI_INFERENCE_DEVICE"] = "cpu"
env["PFI_P10_7_DEVICE"] = "cpu"

command = [
    "python",
    str(REPO / "scripts" / "p10_9_real_checkpoint_smoke.py"),
    "--subarticular", str(P10_6),
    "--disc-multitask", str(P10_7),
    "--output", str(OUTPUT),
]

completed = subprocess.run(
    command,
    cwd=str(REPO),
    env=env,
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        f"REAL_CHECKPOINT_SMOKE_FAILED rc={completed.returncode}"
    )

print("Resultado guardado en:", OUTPUT)


{
  "schemaVersion": "pfi.p10-9.real-checkpoint-smoke.v1",
  "trainingExecuted": false,
  "sealedTestAccessed": false,
  "patientDataUsed": false,
  "subarticular": {
    "status": "PASS",
    "checkpointSha256": "d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a",
    "predictedSeverity": "severe",
    "probabilitySum": 1.0,
    "humanReviewRequired": true,
    "notClinicalDiagnosis": true
  },
  "discMultitask": {
    "status": "PASS",
    "checkpointSha256": "16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293",
    "findingCount": 8,
    "findingTypes": [
      "pfirrmann_grade",
      "modic_change",
      "upper_endplate_change",
      "lower_endplate_change",
      "spondylolisthesis",
      "disc_herniation",
      "disc_narrowing",
      "disc_bulging"
    ],
    "humanReviewRequired": true,
    "notClinicalDiagnosis": true,
    "automaticDiscLocalizationValidated": false
  },
  "realCheckpointForwardValidated": true,
  "automaticDiscLocalizationRea

In [6]:
import json

result = json.loads(OUTPUT.read_text(encoding="utf-8"))

assert result["schemaVersion"] == "pfi.p10-9.real-checkpoint-smoke.v1"
assert result["trainingExecuted"] is False
assert result["sealedTestAccessed"] is False
assert result["patientDataUsed"] is False
assert result["realCheckpointForwardValidated"] is True
assert result["subarticular"]["status"] == "PASS"
assert result["discMultitask"]["status"] == "PASS"
assert result["discMultitask"]["findingCount"] == 8
assert result["automaticDiscLocalizationRealStudyValidated"] is False
assert result["humanReviewRequired"] is True
assert result["notClinicalDiagnosis"] is True

print(json.dumps(result, indent=2, ensure_ascii=False))
print("P10_9_REAL_CHECKPOINT_SMOKE_COMPLETE")


{
  "schemaVersion": "pfi.p10-9.real-checkpoint-smoke.v1",
  "trainingExecuted": false,
  "sealedTestAccessed": false,
  "patientDataUsed": false,
  "subarticular": {
    "status": "PASS",
    "checkpointSha256": "d41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a",
    "predictedSeverity": "severe",
    "probabilitySum": 1.0,
    "humanReviewRequired": true,
    "notClinicalDiagnosis": true
  },
  "discMultitask": {
    "status": "PASS",
    "checkpointSha256": "16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293",
    "findingCount": 8,
    "findingTypes": [
      "pfirrmann_grade",
      "modic_change",
      "upper_endplate_change",
      "lower_endplate_change",
      "spondylolisthesis",
      "disc_herniation",
      "disc_narrowing",
      "disc_bulging"
    ],
    "humanReviewRequired": true,
    "notClinicalDiagnosis": true,
    "automaticDiscLocalizationValidated": false
  },
  "realCheckpointForwardValidated": true,
  "automaticDiscLocalizationRea

## Resultado esperado

El gate solo se considera verde si la última línea es:

`P10_9_REAL_CHECKPOINT_SMOKE_COMPLETE`

Esto valida el **forward real de los checkpoints congelados** sobre entradas
sintéticas. Todavía **no** valida la localización automática sobre un estudio real
ni el E2E Backend ↔ AI; esos son gates posteriores.
